# **1. Setup & Libraries**

In [1]:
!pip install matplotlib-scalebar
!pip install contextily

# For post-hoc seasonal tests
!pip install scikit-posthocs

!pip install pymannkendall


In [2]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.3 MB/s eta 0:00:00


In [3]:
!pip install matplotlib-scalebar

In [4]:
import os
import glob
import calendar

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.ticker import ScalarFormatter
from matplotlib_scalebar.scalebar import ScaleBar
import seaborn as sns
from scipy.stats import mannwhitneyu, kruskal, wilcoxon


# **2. Load Dataset**

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# file_path = r"C:\Users\User\Documents\Data\Final processed 3-hourly rainfall data with missing value.csv"
# df_rain = pd.read_csv(file_path)
# df_rain

In [7]:
file_path = "/content/drive/MyDrive/Rainfall/Data/Final processed 3-hourly rainfall data with missing value.csv"
df_rain = pd.read_csv(file_path)
df_rain

,StationID,StationName,Latitude,Longitude,Primary_Region,Secondary_Region,Datetime,Year,Month,Day,Time,Season,DewPointTemperature,StationLevelPressure,SP,DR,Humidity,Rainfall
0,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 00:00:00,2003,1,1,0,Winter,12.0,1009.7,0.0,0.0,91.0,0.0
1,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 03:00:00,2003,1,1,3,Winter,13.0,1011.3,0.0,0.0,74.0,0.0
2,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 06:00:00,2003,1,1,6,Winter,15.0,1011.2,4.0,31.0,57.0,0.0
3,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 09:00:00,2003,1,1,9,Winter,9.0,1010.3,2.0,5.0,35.0,0.0
4,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 12:00:00,2003,1,1,12,Winter,13.0,1010.4,0.0,0.0,61.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2121051,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-12-31 09:00:00,2023,12,31,9,Winter,14.0,NaN,NaN,NaN,50.2,0.0
2121052,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-12-31 12:00:00,2023,12,31,12,Winter,15.7,NaN,NaN,NaN,71.5,0.0
2121053,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-12-31 15:00:00,2023,12,31,15,Winter,15.6,NaN,NaN,NaN,78.4,0.0
2121054,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-12-31 18:00:00,2023,12,31,18,Winter,14.8,NaN,NaN,NaN,84.2,0.0


In [8]:
# REMOVE THE LATE STATION (Example ID: 10325)
# Replace 10325 with the actual ID of the 2008 station
stations_to_drop = [11921]
df_rain = df_rain[~df_rain["StationID"].isin(stations_to_drop)].reset_index(drop=True)
df_rain


,StationID,StationName,Latitude,Longitude,Primary_Region,Secondary_Region,Datetime,Year,Month,Day,Time,Season,DewPointTemperature,StationLevelPressure,SP,DR,Humidity,Rainfall
0,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 00:00:00,2003,1,1,0,Winter,12.0,1009.7,0.0,0.0,91.0,0.0
1,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 03:00:00,2003,1,1,3,Winter,13.0,1011.3,0.0,0.0,74.0,0.0
2,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 06:00:00,2003,1,1,6,Winter,15.0,1011.2,4.0,31.0,57.0,0.0
3,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 09:00:00,2003,1,1,9,Winter,9.0,1010.3,2.0,5.0,35.0,0.0
4,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 12:00:00,2003,1,1,12,Winter,13.0,1010.4,0.0,0.0,61.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2074299,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-12-31 09:00:00,2023,12,31,9,Winter,14.0,NaN,NaN,NaN,50.2,0.0
2074300,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-12-31 12:00:00,2023,12,31,12,Winter,15.7,NaN,NaN,NaN,71.5,0.0
2074301,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-12-31 15:00:00,2023,12,31,15,Winter,15.6,NaN,NaN,NaN,78.4,0.0
2074302,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-12-31 18:00:00,2023,12,31,18,Winter,14.8,NaN,NaN,NaN,84.2,0.0


# **Hyperparameter Training**

In [ ]:
# ============================================================
# SCRIPT 1: STATION-WISE CATBOOST - HYPERPARAMETER TUNING
# ============================================================

import os, random, math, json
import numpy as np
import pandas as pd
import joblib

from catboost import CatBoostRegressor, CatBoostClassifier

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import average_precision_score, roc_auc_score

# ------------------------------------------------------------
# 0) Reproducibility + Device
# ------------------------------------------------------------
def set_seed(seed=42, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42, deterministic=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

TUNING_SAVE_DIR =r"C:\Users\pc\Documents\Rainfall_Training_Research_Lab\Baselines\CatBoost\Tuning_v1"
os.makedirs(TUNING_SAVE_DIR, exist_ok=True)
print("CatBoost Tuning directory:", TUNING_SAVE_DIR)

# ============================================================
# 1) Core Utilities
# ============================================================
class NaNIgnoringStandardScaler:
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        self.mean_ = np.nanmean(X, axis=0)
        self.std_  = np.nanstd(X, axis=0)
        self.std_  = np.where(self.std_ < self.eps, 1.0, self.std_)
        return self

    def transform(self, X):
        return (X - self.mean_) / self.std_

def pinball_loss(y_hat_q, y_true, q, mask=None):
    e = y_true - y_hat_q
    loss = torch.maximum((q - 1.0) * e, q * e)
    if mask is not None:
        loss = loss * mask
        return loss.sum() / mask.sum().clamp_min(1.0)
    return loss.mean()

def crps_from_quantiles(y_hat, y_true, quantiles, mask=None):
    losses = [pinball_loss(y_hat[..., k], y_true, q, mask=mask) for k, q in enumerate(quantiles)]
    return 2.0 * torch.stack(losses).mean()

def brier_score(probs, targets, mask=None):
    loss = (probs - targets) ** 2
    if mask is not None:
        loss = loss * mask
        return loss.sum() / mask.sum().clamp_min(1.0)
    return loss.mean()

def safe_div(a, b, eps=1e-12):
    return a / (b + eps)

def event_metrics_binary(probs, y_true, mask, thresholds=(0.1, 0.2, 0.3, 0.4, 0.5)):
    valid = mask > 0.5
    if valid.sum() < 50:
        return {"n_valid": int(valid.sum()), "pr_auc": np.nan, "roc_auc": np.nan, "by_thr": {thr: {"POD": np.nan, "FAR": np.nan, "CSI": np.nan} for thr in thresholds}}
    p, y = probs[valid], y_true[valid]
    pr_auc = average_precision_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    try: roc_auc = roc_auc_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    except: roc_auc = np.nan
    by_thr = {}
    for thr in thresholds:
        yhat = (p >= thr).astype(np.float32)
        TP, FP, FN = ((yhat == 1) & (y == 1)).sum(), ((yhat == 1) & (y == 0)).sum(), ((yhat == 0) & (y == 1)).sum()
        by_thr[thr] = {"POD": float(safe_div(TP, TP + FN)), "FAR": float(safe_div(FP, TP + FP)), "CSI": float(safe_div(TP, TP + FP + FN))}
    return {"n_valid": int(valid.sum()), "pr_auc": float(pr_auc) if pr_auc == pr_auc else np.nan, "roc_auc": float(roc_auc) if roc_auc == roc_auc else np.nan, "by_thr": by_thr}

def make_splits(T, train_frac=0.7, val_frac=0.15):
    return int(T * train_frac), int(T * (train_frac + val_frac))

def scale_inputs_train_only(X_raw, train_end):
    T, N, F = X_raw.shape
    X_flat = X_raw.reshape(T*N, F)
    scaler = NaNIgnoringStandardScaler().fit(X_flat[:train_end*N])
    return scaler.transform(X_flat).reshape(T, N, F).astype(np.float32), scaler

def compute_thresholds_train(Y, M, train_end, q=0.95):
    Y_train, M_train, N = Y[:train_end], M[:train_end], Y.shape[1]
    thr = np.zeros(N, dtype=np.float32)
    global_vals = Y_train[M_train > 0.5]
    global_fallback = np.nanpercentile(global_vals, q*100) if global_vals.size > 0 else 0.0
    for j in range(N):
        vals = Y_train[:, j][M_train[:, j] > 0.5]
        thr[j] = np.nanpercentile(vals, q*100) if len(vals) >= 100 else global_fallback
    return thr

def compute_acc24(Y_rain, M_y, H_out=8):
    T, N = Y_rain.shape
    Acc, Mask = np.full((T, N), np.nan, dtype=np.float32), np.zeros((T, N), dtype=np.float32)
    for t in range(T - H_out):
        window, wmask = Y_rain[t+1:t+1+H_out], M_y[t+1:t+1+H_out]
        ok = (wmask.sum(axis=0) == H_out)
        Mask[t, ok], Acc[t, ok] = 1.0, window[:, ok].sum(axis=0)
    return Acc, Mask

# ============================================================
# 2) Preprocessing & Dataset
# ============================================================
def prepare_full_grid(df_rain):
    df = df_rain.copy()
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.sort_values(["Datetime", "StationID"]).reset_index(drop=True)
    dr_rad = np.deg2rad(df["DR"].astype(np.float32).to_numpy())
    df["DR_sin"], df["DR_cos"] = np.sin(dr_rad), np.cos(dr_rad)
    stations = df[["StationID", "StationName", "Latitude", "Longitude"]].drop_duplicates().sort_values("StationID").reset_index(drop=True)
    stations["node_index"] = np.arange(len(stations))
    N, times = len(stations), np.sort(df["Datetime"].unique())
    T = len(times)

    full_index = pd.MultiIndex.from_product([times, stations["StationID"].values], names=["Datetime", "StationID"])
    meteo_cols = ["DewPointTemperature", "StationLevelPressure", "SP", "Humidity", "Rainfall", "DR_sin", "DR_cos"]
    X_raw = df.set_index(["Datetime", "StationID"])[meteo_cols].sort_index().reindex(full_index).values.reshape(T, N, len(meteo_cols)).astype(np.float32)
    M_in = (~np.isnan(X_raw)).astype(np.float32)
    Y_rain, M_y = X_raw[..., meteo_cols.index("Rainfall")], M_in[..., meteo_cols.index("Rainfall")]

    dt_index = pd.DatetimeIndex(times)
    hour, month = dt_index.hour.astype(np.float32), dt_index.month.astype(np.float32)
    time_feats = np.repeat(np.stack([np.sin(2*np.pi*(hour/24.0)), np.cos(2*np.pi*(hour/24.0)), np.sin(2*np.pi*((month-1)/12.0)), np.cos(2*np.pi*((month-1)/12.0))], axis=-1).astype(np.float32)[:, None, :], N, axis=1)

    season_by_time = df.groupby("Datetime")["Season"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0]).reindex(times).astype(str).values
    unique_seasons = sorted(pd.unique(season_by_time))
    season_to_id = {s: i for i, s in enumerate(unique_seasons)}

    return {"stations": stations, "times": times, "X_raw": X_raw, "M_in": M_in, "Y_rain": Y_rain, "M_y": M_y, "meteo_cols": meteo_cols, "time_feats": time_feats, "season_ids": np.array([season_to_id[s] for s in season_by_time], dtype=np.int64), "unique_seasons": unique_seasons}

class BanglaRainDataset(Dataset):
    def __init__(self, X_scaled, M_in, time_feats, Y_rain, M_y, Acc24, MaskAcc24, season_ids, thr3h, thrAcc24, t_start, t_end, T_in=16, H_out=8, peak_min_obs=None):
        self.X_scaled, self.M_in, self.time_feats = X_scaled, M_in, time_feats
        self.Y_rain, self.M_y, self.Acc24, self.MaskAcc24 = Y_rain, M_y, Acc24, MaskAcc24
        self.season_ids, self.thr3h, self.thrAcc24 = season_ids, thr3h, thrAcc24
        self.T_in, self.H_out = T_in, H_out
        self.peak_min_obs = peak_min_obs if peak_min_obs is not None else (H_out // 2)
        self.indices = list(range(t_start + (T_in - 1), t_end - H_out))

    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        t = self.indices[idx]
        x_all = np.concatenate([np.nan_to_num(self.X_scaled[t-self.T_in+1:t+1], nan=0.0).astype(np.float32), self.M_in[t-self.T_in+1:t+1].astype(np.float32), self.time_feats[t-self.T_in+1:t+1]], axis=-1)
        y_win, m_win = self.Y_rain[t+1:t+1+self.H_out], self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        y_log = np.log1p(np.nan_to_num(y_win, nan=0.0)).astype(np.float32)
        m_next = self.M_y[t+1].astype(np.float32)
        flash = ((self.Y_rain[t+1] >= self.thr3h) & (m_next > 0.5)).astype(np.float32)
        mpeak = (m_win.sum(axis=0) >= self.peak_min_obs).astype(np.float32)
        peak24 = ((np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0) >= self.thr3h) & (mpeak > 0.5)).astype(np.float32)
        macc = self.MaskAcc24[t].astype(np.float32)
        acc24 = ((self.Acc24[t] >= self.thrAcc24) & (macc > 0.5)).astype(np.float32)
        return torch.from_numpy(x_all), torch.tensor(int(self.season_ids[t]), dtype=torch.long), torch.from_numpy(y_log), torch.from_numpy(m_win), torch.from_numpy(flash), torch.from_numpy(m_next), torch.from_numpy(peak24), torch.from_numpy(mpeak), torch.from_numpy(acc24), torch.from_numpy(macc)

# ============================================================
# 3) Build Tabular + Evaluate
# ============================================================
def build_data_objects(df_rain, T_in=16, H_out=8, train_frac=0.7, val_frac=0.15):
    prep = prepare_full_grid(df_rain)
    T = len(prep["times"])
    train_end, val_end = make_splits(T, train_frac, val_frac)
    X_scaled, scaler = scale_inputs_train_only(prep["X_raw"], train_end)
    thr3h = compute_thresholds_train(prep["Y_rain"], prep["M_y"], train_end, q=0.95)
    Acc24, MaskAcc24 = compute_acc24(prep["Y_rain"], prep["M_y"], H_out)
    thrAcc24 = compute_thresholds_train(Acc24, MaskAcc24, train_end, q=0.95)

    ds_train = BanglaRainDataset(X_scaled, prep["M_in"], prep["time_feats"], prep["Y_rain"], prep["M_y"], Acc24, MaskAcc24, prep["season_ids"], thr3h, thrAcc24, 0, train_end, T_in, H_out)
    ds_val = BanglaRainDataset(X_scaled, prep["M_in"], prep["time_feats"], prep["Y_rain"], prep["M_y"], Acc24, MaskAcc24, prep["season_ids"], thr3h, thrAcc24, train_end, val_end, T_in, H_out)
    ds_test = BanglaRainDataset(X_scaled, prep["M_in"], prep["time_feats"], prep["Y_rain"], prep["M_y"], Acc24, MaskAcc24, prep["season_ids"], thr3h, thrAcc24, val_end, T, T_in, H_out)
    return prep, scaler, (thr3h, thrAcc24), (Acc24, MaskAcc24), (train_end, val_end, T), (ds_train, ds_val, ds_test)

def make_loaders(ds_train, ds_val, ds_test, batch_size=256):
    return DataLoader(ds_train, batch_size=batch_size, shuffle=True, drop_last=True), DataLoader(ds_val, batch_size=batch_size), DataLoader(ds_test, batch_size=batch_size)

def build_tabular_from_loader(loader):
    X_reg_list, y_reg_list, station_reg_list, lead_reg_list, mask_reg_list = [], [], [], [], []
    X_clf_list, station_clf_list, flash_y_list, flash_m_list, peak_y_list, peak_m_list, acc_y_list, acc_m_list = [], [], [], [], [], [], [], []

    for x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc in loader:
        x_np, y_mm, my_np = x[:, -1, :, :].numpy(), torch.expm1(y_log).clamp_min(0.0).numpy(), my.numpy()
        flash_np, mflash_np, peak_np, mpeak_np, acc_np, macc_np = flash.numpy(), mflash.numpy(), peak.numpy(), mpeak.numpy(), acc.numpy(), macc.numpy()
        B, H, N = y_mm.shape

        for b in range(B):
            for h in range(H):
                mask_h = my_np[b, h, :] > 0.5
                if not mask_h.any(): continue
                X_bh, y_bh = x_np[b], y_mm[b, h, :]
                for j in range(N):
                    if mask_h[j]:
                        X_reg_list.append(X_bh[j]); y_reg_list.append(y_bh[j]); station_reg_list.append(j); lead_reg_list.append(h); mask_reg_list.append(1.0)
            for j in range(N):
                if mflash_np[b, j] > 0.5:
                    X_clf_list.append(x_np[b, j]); station_clf_list.append(j)
                    flash_y_list.append(flash_np[b, j]); peak_y_list.append(peak_np[b, j]); acc_y_list.append(acc_np[b, j])
                    flash_m_list.append(mflash_np[b, j]); peak_m_list.append(mpeak_np[b, j]); acc_m_list.append(macc_np[b, j])

    return {
        "X_reg": np.array(X_reg_list, dtype=np.float32), "y_reg": np.array(y_reg_list, dtype=np.float32), "station_reg": np.array(station_reg_list, dtype=np.int64),
        "X_clf": np.array(X_clf_list, dtype=np.float32), "station_clf": np.array(station_clf_list, dtype=np.int64),
        "flash_y": np.array(flash_y_list, dtype=np.float32), "flash_m": np.array(flash_m_list, dtype=np.float32),
        "peak_y": np.array(peak_y_list, dtype=np.float32), "peak_m": np.array(peak_m_list, dtype=np.float32),
        "acc_y": np.array(acc_y_list, dtype=np.float32), "acc_m": np.array(acc_m_list, dtype=np.float32),
    }

class StationWiseCatBoostBaseline(nn.Module):
    def __init__(self, cat_regressors, cat_flash, cat_peak, cat_acc, num_stations, H_out, quantiles):
        super().__init__()
        self.cat_regressors, self.cat_flash, self.cat_peak, self.cat_acc = cat_regressors, cat_flash, cat_peak, cat_acc
        self.N, self.H_out, self.quantiles = num_stations, H_out, list(quantiles)
        self.K = len(self.quantiles)

    def forward(self, x, regime_id):
        B, T, N, F = x.shape
        x_last = x[:, -1, :, :].detach().cpu().numpy()
        q_mm, flash_prob, peak_prob, acc_prob = np.zeros((B, self.H_out, N, self.K), dtype=np.float32), np.zeros((B, N), dtype=np.float32), np.zeros((B, N), dtype=np.float32), np.zeros((B, N), dtype=np.float32)

        for j in range(N):
            X_j = x_last[:, j, :]
            for k, q in enumerate(self.quantiles):
                model = self.cat_regressors.get((j, q), None)
                q_mm[:, :, j, k] = np.zeros(B, dtype=np.float32)[:, None] if model is None else model.predict(X_j)[:, None]

            m_flash = self.cat_flash.get(j, None)
            flash_prob[:, j] = 0.0 if m_flash is None else m_flash.predict_proba(X_j)[:, 1]

            m_peak = self.cat_peak.get(j, None)
            peak_prob[:, j] = 0.0 if m_peak is None else m_peak.predict_proba(X_j)[:, 1]

            m_acc = self.cat_acc.get(j, None)
            acc_prob[:, j] = 0.0 if m_acc is None else m_acc.predict_proba(X_j)[:, 1]

        q_log = torch.log1p(torch.from_numpy(q_mm).to(device)).clamp_min(0.0)
        def prob_to_logits(p):
            p = torch.clamp(p, 1e-6, 1.0 - 1e-6)
            return torch.log(p / (1.0 - p))
        return q_log, prob_to_logits(torch.from_numpy(flash_prob).to(device)), prob_to_logits(torch.from_numpy(peak_prob).to(device)), prob_to_logits(torch.from_numpy(acc_prob).to(device))

@torch.no_grad()
def evaluate(model, loader, quantiles, thresholds=(0.1, 0.2, 0.3, 0.5)):
    model.eval()
    total_crps_log, total_crps_mm, total_brier_flash, total_brier_peak, total_brier_acc, nb = 0.0, 0.0, 0.0, 0.0, 0.0, 0
    k50 = int(np.argmin(np.abs(np.array(list(quantiles), dtype=np.float32) - 0.5)))
    flash_p_list, flash_y_list, flash_m_list, peak_p_list, peak_y_list, peak_m_list, acc_p_list, acc_y_list, acc_m_list = [], [], [], [], [], [], [], [], []

    for x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc in loader:
        x, reg, y_log, my = x.to(device), reg.to(device), y_log.to(device), my.to(device)
        flash, mflash, peak, mpeak, acc, macc = flash.to(device), mflash.to(device), peak.to(device), mpeak.to(device), acc.to(device), macc.to(device)
        q_hat, flash_logits, peak_logits, acc_logits = model(x, reg)

        total_crps_log += crps_from_quantiles(q_hat, y_log, quantiles, mask=my).item()
        q_mm, y_mm = torch.expm1(q_hat).clamp_min(0.0), torch.expm1(y_log).clamp_min(0.0)
        total_crps_mm += crps_from_quantiles(q_mm, y_mm, quantiles, mask=my).item()

        p_flash, p_peak, p_acc = torch.sigmoid(flash_logits), torch.sigmoid(peak_logits), torch.sigmoid(acc_logits)
        total_brier_flash += brier_score(p_flash, flash, mask=mflash).item()
        total_brier_peak += brier_score(p_peak, peak, mask=mpeak).item()
        total_brier_acc += brier_score(p_acc, acc, mask=macc).item()

        flash_p_list.append(p_flash.detach().cpu().reshape(-1)); flash_y_list.append(flash.detach().cpu().reshape(-1)); flash_m_list.append(mflash.detach().cpu().reshape(-1))
        peak_p_list.append(p_peak.detach().cpu().reshape(-1)); peak_y_list.append(peak.detach().cpu().reshape(-1)); peak_m_list.append(mpeak.detach().cpu().reshape(-1))
        acc_p_list.append(p_acc.detach().cpu().reshape(-1)); acc_y_list.append(acc.detach().cpu().reshape(-1)); acc_m_list.append(macc.detach().cpu().reshape(-1))
        nb += 1

    flash_p, flash_y, flash_m = torch.cat(flash_p_list).numpy(), torch.cat(flash_y_list).numpy(), torch.cat(flash_m_list).numpy()
    peak_p, peak_y, peak_m = torch.cat(peak_p_list).numpy(), torch.cat(peak_y_list).numpy(), torch.cat(peak_m_list).numpy()
    acc_p, acc_y, acc_m = torch.cat(acc_p_list).numpy(), torch.cat(acc_y_list).numpy(), torch.cat(acc_m_list).numpy()

    return {
        "CRPS_log": total_crps_log / max(nb, 1),
        "CRPS_mm": total_crps_mm / max(nb, 1),
        "Brier_flash": total_brier_flash / max(nb, 1),
        "Brier_peak": total_brier_peak / max(nb, 1),
        "Brier_acc": total_brier_acc / max(nb, 1),
        "Flash_metrics": event_metrics_binary(flash_p, flash_y, flash_m, thresholds=thresholds),
        "Peak24_metrics": event_metrics_binary(peak_p, peak_y, peak_m, thresholds=thresholds),
        "Acc24_metrics": event_metrics_binary(acc_p, acc_y, acc_m, thresholds=thresholds),
    }

# ============================================================
# 4) CatBoost Training Engine
# ============================================================
def sample_hparams(rng):
    return {
        "learning_rate": float(10 ** rng.uniform(math.log10(1e-3), math.log10(3e-1))),
        "iterations": int(rng.randint(50, 250)),
        "depth": int(rng.randint(4, 9)),
        "l2_leaf_reg": float(10 ** rng.uniform(-4, 2)),
        "subsample": float(rng.uniform(0.5, 1.0)),
        "rsm": float(rng.uniform(0.5, 1.0)),
        "min_data_in_leaf": int(rng.randint(20, 100))
    }

def train_station_regressors(X_reg, y_reg, station_reg, params_base, quantiles, num_stations):
    cat_regressors = {}
    for j in range(num_stations):
        mask_j = (station_reg == j)
        if not np.any(mask_j): continue
        X_j, y_j = X_reg[mask_j], y_reg[mask_j]
        if X_j.shape[0] < 50: continue

        for q in quantiles:
            model = CatBoostRegressor(loss_function=f'Quantile:alpha={q}', bootstrap_type='Bernoulli', **params_base)
            model.fit(X_j, y_j, verbose=False)
            cat_regressors[(j, q)] = model
    return cat_regressors

def train_station_classifiers(tab, num_stations, params_base):
    X_clf, station_clf = tab["X_clf"], tab["station_clf"]

    def train_one_target(y, m):
        models = {}
        for j in range(num_stations):
            mask_station = (station_clf == j)
            if not np.any(mask_station): continue
            X_j, y_j, m_j = X_clf[mask_station], y[mask_station], m[mask_station]
            valid = (m_j > 0.5)
            X_j, y_j = X_j[valid], y_j[valid]

            if X_j.shape[0] < 20:
                X_dummy, y_dummy = np.zeros((50, X_clf.shape[1]), dtype=np.float32), np.zeros(50, dtype=np.float32)
                model = CatBoostClassifier(loss_function='Logloss', bootstrap_type='Bernoulli', iterations=10, learning_rate=0.1, depth=4, verbose=False)
                model.fit(X_dummy, y_dummy)
                models[j] = model
                continue

            model = CatBoostClassifier(loss_function='Logloss', bootstrap_type='Bernoulli', **params_base)
            model.fit(X_j, y_j, verbose=False)
            models[j] = model
        return models

    return train_one_target(tab["flash_y"], tab["flash_m"]), train_one_target(tab["peak_y"], tab["peak_m"]), train_one_target(tab["acc_y"], tab["acc_m"])

# ============================================================
# 5) Main Tuning Script (UPDATED: RESUME-SAFE)
# ============================================================
def tune_catboost_stationwise(df_rain, T_in=16, H_out=8, train_frac=0.7, val_frac=0.15, quantiles=(0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95), n_trials=15):
    prep, scaler, (thr3h, thrAcc24), (Acc24, MaskAcc24), splits, datasets = build_data_objects(df_rain, T_in, H_out, train_frac, val_frac)
    train_end, val_end, T_total = splits
    ds_train, ds_val, ds_test = datasets
    train_loader, val_loader, _ = make_loaders(ds_train, ds_val, ds_test, batch_size=256)
    N_stations = len(prep["stations"])

    tab_train = build_tabular_from_loader(train_loader)
    rng = np.random.RandomState(42)

    # --- RESUME SAFE LOGIC ---
    results_csv = os.path.join(TUNING_SAVE_DIR, "catboost_tuning_results.csv")
    best_score, best_trial, trial_rows = float("inf"), None, []
    start_trial = 1

    if os.path.exists(results_csv):
        print(f"\n[Resume] Found existing tuning results. Resuming...")
        existing_df = pd.read_csv(results_csv)
        trial_rows = existing_df.to_dict('records')
        start_trial = len(trial_rows) + 1
        best_score = existing_df['val_CRPS_log'].min()

        best_json = os.path.join(TUNING_SAVE_DIR, "best_trial.json")
        if os.path.exists(best_json):
            with open(best_json, "r") as f:
                best_trial = json.load(f)

        print(f"[Resume] Starting at Trial {start_trial}. Current Best CRPS: {best_score:.4f}")

    if start_trial > n_trials:
        print("\n[CatBoost Tuning] Tuning already completed to specified n_trials.")
        return best_trial

    for trial in range(start_trial, n_trials + 1):
        print(f"\n[CatBoost Tuning] Trial {trial}/{n_trials}")
        hp = sample_hparams(rng)

        params_base = {
            "iterations": hp["iterations"],
            "learning_rate": hp["learning_rate"],
            "depth": hp["depth"],
            "l2_leaf_reg": hp["l2_leaf_reg"],
            "subsample": hp["subsample"],
            "rsm": hp["rsm"],
            "min_data_in_leaf": hp["min_data_in_leaf"],
            "random_seed": 42 + trial, # Changing seed slightly per trial ensures variety
            "verbose": True,
            "thread_count": -1
        }

        cat_reg = train_station_regressors(tab_train["X_reg"], tab_train["y_reg"], tab_train["station_reg"], params_base, quantiles, N_stations)
        cat_flash, cat_peak, cat_acc = train_station_classifiers(tab_train, N_stations, params_base)

        model = StationWiseCatBoostBaseline(cat_reg, cat_flash, cat_peak, cat_acc, N_stations, H_out, quantiles).to(device)
        val_scores = evaluate(model, val_loader, quantiles)
        val_CRPS_log = float(val_scores["CRPS_log"])
        print(f"  -> val_CRPS_log = {val_CRPS_log:.4f}")

        row = {"trial": trial, "val_CRPS_log": val_CRPS_log}; row.update(hp)
        trial_rows.append(row)
        pd.DataFrame(trial_rows).to_csv(results_csv, index=False)

        if val_CRPS_log < best_score - 1e-6:
            best_score = val_CRPS_log
            best_trial = row
            joblib.dump({"cat_reg": cat_reg, "cat_flash": cat_flash, "cat_peak": cat_peak, "cat_acc": cat_acc}, os.path.join(TUNING_SAVE_DIR, "best_models.pkl"))
            with open(os.path.join(TUNING_SAVE_DIR, "best_trial.json"), "w") as f: json.dump(best_trial, f, indent=2)
            print(f"  -> New best trial (CRPS_log = {best_score:.4f})")

    return best_trial

# >>> EXECUTE TUNING HERE <<<
best_trial = tune_catboost_stationwise(df_rain=df_rain, n_trials=15)

Using device: cpu
CatBoost Tuning directory: C:\Users\pc\Documents\Rainfall_Training_Research_Lab\Baselines\CatBoost\Tuning_v1


C:\Users\pc\AppData\Local\Temp\ipykernel_21536\1067779594.py:169: RuntimeWarning: All-NaN slice encountered
  peak24 = ((np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0) >= self.thr3h) & (mpeak > 0.5)).astype(np.float32)



[Resume] Found existing tuning results. Resuming...
[Resume] Starting at Trial 7. Current Best CRPS: 0.1302

[CatBoost Tuning] Trial 7/15


C:\Users\pc\AppData\Local\Temp\ipykernel_21536\1067779594.py:169: RuntimeWarning: All-NaN slice encountered
  peak24 = ((np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0) >= self.thr3h) & (mpeak > 0.5)).astype(np.float32)


  -> val_CRPS_log = 0.1355

[CatBoost Tuning] Trial 8/15


...

  -> val_CRPS_log = 0.1405

[CatBoost Tuning] Trial 14/15


C:\Users\pc\AppData\Local\Temp\ipykernel_21536\1067779594.py:169: RuntimeWarning: All-NaN slice encountered
  peak24 = ((np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0) >= self.thr3h) & (mpeak > 0.5)).astype(np.float32)


  -> val_CRPS_log = 0.1369

[CatBoost Tuning] Trial 15/15


C:\Users\pc\AppData\Local\Temp\ipykernel_21536\1067779594.py:169: RuntimeWarning: All-NaN slice encountered
  peak24 = ((np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0) >= self.thr3h) & (mpeak > 0.5)).astype(np.float32)


  -> val_CRPS_log = 0.1461


# **Model Training**

In [ ]:
!pip install catboost

In [ ]:
# ============================================================
# BASELINE: Station-Wise CatBoost - FINAL TRAINING
# ------------------------------------------------------------
# - Assumes df_rain is already loaded in memory (DataFrame)
# - Loads best hyperparameters from tuning
# - Trains final model using combined Train + Val datasets
# - Saves to FINAL_SAVE_DIR:
#     * final_training_summary.csv (1 row, as there are no epochs)
#     * final_best_model.pkl
# ============================================================

import os, json, random, math
import numpy as np
import pandas as pd
import joblib

from catboost import CatBoostRegressor, CatBoostClassifier

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from sklearn.metrics import average_precision_score, roc_auc_score

# ----------------------------
# 0) Reproducibility + Device
# ----------------------------
def set_seed(seed=42, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42, deterministic=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

TUNING_SAVE_DIR = r"C:\Users\User\Documents\CatBoost\Tuning_v1"
FINAL_SAVE_DIR  = r"C:\Users\User\Documents\CatBoost\Final_Run\v2"

os.makedirs(TUNING_SAVE_DIR, exist_ok=True)
os.makedirs(FINAL_SAVE_DIR, exist_ok=True)

print("Tuning directory :", TUNING_SAVE_DIR)
print("Final save dir   :", FINAL_SAVE_DIR)

# ============================================================
# 1) Core Utilities
# ============================================================
class NaNIgnoringStandardScaler:
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        self.mean_ = np.nanmean(X, axis=0)
        self.std_  = np.nanstd(X, axis=0)
        self.std_  = np.where(self.std_ < self.eps, 1.0, self.std_)
        return self

    def transform(self, X):
        return (X - self.mean_) / self.std_

def pinball_loss(y_hat_q, y_true, q, mask=None):
    e = y_true - y_hat_q
    loss = torch.maximum((q - 1.0) * e, q * e)
    if mask is not None:
        loss = loss * mask
        return loss.sum() / mask.sum().clamp_min(1.0)
    return loss.mean()

def crps_from_quantiles(y_hat, y_true, quantiles, mask=None):
    losses = [pinball_loss(y_hat[..., k], y_true, q, mask=mask) for k, q in enumerate(quantiles)]
    return 2.0 * torch.stack(losses).mean()

def brier_score(probs, targets, mask=None):
    loss = (probs - targets) ** 2
    if mask is not None:
        loss = loss * mask
        return loss.sum() / mask.sum().clamp_min(1.0)
    return loss.mean()

def safe_div(a, b, eps=1e-12):
    return a / (b + eps)

def event_metrics_binary(probs, y_true, mask, thresholds=(0.1, 0.2, 0.3, 0.4, 0.5)):
    valid = mask > 0.5
    if valid.sum() < 50:
        return {"n_valid": int(valid.sum()), "pr_auc": np.nan, "roc_auc": np.nan, "by_thr": {thr: {"POD": np.nan, "FAR": np.nan, "CSI": np.nan} for thr in thresholds}}
    p, y = probs[valid], y_true[valid]
    pr_auc = average_precision_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    try: roc_auc = roc_auc_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    except: roc_auc = np.nan
    by_thr = {}
    for thr in thresholds:
        yhat = (p >= thr).astype(np.float32)
        TP, FP, FN = ((yhat == 1) & (y == 1)).sum(), ((yhat == 1) & (y == 0)).sum(), ((yhat == 0) & (y == 1)).sum()
        by_thr[thr] = {"POD": float(safe_div(TP, TP + FN)), "FAR": float(safe_div(FP, TP + FP)), "CSI": float(safe_div(TP, TP + FP + FN))}
    return {"n_valid": int(valid.sum()), "pr_auc": float(pr_auc) if pr_auc == pr_auc else np.nan, "roc_auc": float(roc_auc) if roc_auc == roc_auc else np.nan, "by_thr": by_thr}

def make_splits(T, train_frac=0.7, val_frac=0.15):
    return int(T * train_frac), int(T * (train_frac + val_frac))

def scale_inputs_train_only(X_raw, train_end):
    T, N, F = X_raw.shape
    X_flat = X_raw.reshape(T*N, F)
    scaler = NaNIgnoringStandardScaler().fit(X_flat[:train_end*N])
    return scaler.transform(X_flat).reshape(T, N, F).astype(np.float32), scaler

def compute_thresholds_train(Y, M, train_end, q=0.95):
    Y_train, M_train, N = Y[:train_end], M[:train_end], Y.shape[1]
    thr = np.zeros(N, dtype=np.float32)
    global_vals = Y_train[M_train > 0.5]
    global_fallback = np.nanpercentile(global_vals, q*100) if global_vals.size > 0 else 0.0
    for j in range(N):
        vals = Y_train[:, j][M_train[:, j] > 0.5]
        thr[j] = np.nanpercentile(vals, q*100) if len(vals) >= 100 else global_fallback
    return thr

def compute_acc24(Y_rain, M_y, H_out=8):
    T, N = Y_rain.shape
    Acc, Mask = np.full((T, N), np.nan, dtype=np.float32), np.zeros((T, N), dtype=np.float32)
    for t in range(T - H_out):
        window, wmask = Y_rain[t+1:t+1+H_out], M_y[t+1:t+1+H_out]
        ok = (wmask.sum(axis=0) == H_out)
        Mask[t, ok], Acc[t, ok] = 1.0, window[:, ok].sum(axis=0)
    return Acc, Mask

# ============================================================
# 2) Preprocessing & Dataset
# ============================================================
def prepare_full_grid(df_rain):
    df = df_rain.copy()
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.sort_values(["Datetime", "StationID"]).reset_index(drop=True)
    dr_rad = np.deg2rad(df["DR"].astype(np.float32).to_numpy())
    df["DR_sin"], df["DR_cos"] = np.sin(dr_rad), np.cos(dr_rad)
    stations = df[["StationID", "StationName", "Latitude", "Longitude"]].drop_duplicates().sort_values("StationID").reset_index(drop=True)
    stations["node_index"] = np.arange(len(stations))
    N, times = len(stations), np.sort(df["Datetime"].unique())
    T = len(times)

    full_index = pd.MultiIndex.from_product([times, stations["StationID"].values], names=["Datetime", "StationID"])
    meteo_cols = ["DewPointTemperature", "StationLevelPressure", "SP", "Humidity", "Rainfall", "DR_sin", "DR_cos"]
    X_raw = df.set_index(["Datetime", "StationID"])[meteo_cols].sort_index().reindex(full_index).values.reshape(T, N, len(meteo_cols)).astype(np.float32)
    M_in = (~np.isnan(X_raw)).astype(np.float32)
    Y_rain, M_y = X_raw[..., meteo_cols.index("Rainfall")], M_in[..., meteo_cols.index("Rainfall")]

    dt_index = pd.DatetimeIndex(times)
    hour, month = dt_index.hour.astype(np.float32), dt_index.month.astype(np.float32)
    time_feats = np.repeat(np.stack([np.sin(2*np.pi*(hour/24.0)), np.cos(2*np.pi*(hour/24.0)), np.sin(2*np.pi*((month-1)/12.0)), np.cos(2*np.pi*((month-1)/12.0))], axis=-1).astype(np.float32)[:, None, :], N, axis=1)

    season_by_time = df.groupby("Datetime")["Season"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0]).reindex(times).astype(str).values
    unique_seasons = sorted(pd.unique(season_by_time))
    season_to_id = {s: i for i, s in enumerate(unique_seasons)}

    return {"stations": stations, "times": times, "X_raw": X_raw, "M_in": M_in, "Y_rain": Y_rain, "M_y": M_y, "meteo_cols": meteo_cols, "time_feats": time_feats, "season_ids": np.array([season_to_id[s] for s in season_by_time], dtype=np.int64), "unique_seasons": unique_seasons}

class BanglaRainDataset(Dataset):
    def __init__(self, X_scaled, M_in, time_feats, Y_rain, M_y, Acc24, MaskAcc24, season_ids, thr3h, thrAcc24, t_start, t_end, T_in=16, H_out=8, peak_min_obs=None):
        self.X_scaled, self.M_in, self.time_feats = X_scaled, M_in, time_feats
        self.Y_rain, self.M_y, self.Acc24, self.MaskAcc24 = Y_rain, M_y, Acc24, MaskAcc24
        self.season_ids, self.thr3h, self.thrAcc24 = season_ids, thr3h, thrAcc24
        self.T_in, self.H_out = T_in, H_out
        self.peak_min_obs = peak_min_obs if peak_min_obs is not None else (H_out // 2)
        self.indices = list(range(t_start + (T_in - 1), t_end - H_out))

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]
        x_all = np.concatenate([np.nan_to_num(self.X_scaled[t-self.T_in+1:t+1], nan=0.0).astype(np.float32), self.M_in[t-self.T_in+1:t+1].astype(np.float32), self.time_feats[t-self.T_in+1:t+1]], axis=-1)
        y_win, m_win = self.Y_rain[t+1:t+1+self.H_out], self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        y_log = np.log1p(np.nan_to_num(y_win, nan=0.0)).astype(np.float32)
        m_next = self.M_y[t+1].astype(np.float32)
        flash = ((self.Y_rain[t+1] >= self.thr3h) & (m_next > 0.5)).astype(np.float32)
        mpeak = (m_win.sum(axis=0) >= self.peak_min_obs).astype(np.float32)

        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)

        peak24 = ((y_peak >= self.thr3h) & (mpeak > 0.5)).astype(np.float32)
        macc = self.MaskAcc24[t].astype(np.float32)
        acc24 = ((self.Acc24[t] >= self.thrAcc24) & (macc > 0.5)).astype(np.float32)
        return torch.from_numpy(x_all), torch.tensor(int(self.season_ids[t]), dtype=torch.long), torch.from_numpy(y_log), torch.from_numpy(m_win), torch.from_numpy(flash), torch.from_numpy(m_next), torch.from_numpy(peak24), torch.from_numpy(mpeak), torch.from_numpy(acc24), torch.from_numpy(macc)

# ============================================================
# 3) Build Tabular
# ============================================================
def build_data_objects(df_rain, T_in=16, H_out=8, train_frac=0.7, val_frac=0.15):
    prep = prepare_full_grid(df_rain)
    T = len(prep["times"])
    train_end, val_end = make_splits(T, train_frac, val_frac)
    X_scaled, scaler = scale_inputs_train_only(prep["X_raw"], train_end)
    thr3h = compute_thresholds_train(prep["Y_rain"], prep["M_y"], train_end, q=0.95)
    Acc24, MaskAcc24 = compute_acc24(prep["Y_rain"], prep["M_y"], H_out)
    thrAcc24 = compute_thresholds_train(Acc24, MaskAcc24, train_end, q=0.95)

    ds_train = BanglaRainDataset(X_scaled, prep["M_in"], prep["time_feats"], prep["Y_rain"], prep["M_y"], Acc24, MaskAcc24, prep["season_ids"], thr3h, thrAcc24, 0, train_end, T_in, H_out)
    ds_val = BanglaRainDataset(X_scaled, prep["M_in"], prep["time_feats"], prep["Y_rain"], prep["M_y"], Acc24, MaskAcc24, prep["season_ids"], thr3h, thrAcc24, train_end, val_end, T_in, H_out)

    return prep, scaler, (thr3h, thrAcc24), (ds_train, ds_val)

def build_tabular_from_loader(loader):
    X_reg_list, y_reg_list, station_reg_list, lead_reg_list, mask_reg_list = [], [], [], [], []
    X_clf_list, station_clf_list, flash_y_list, flash_m_list, peak_y_list, peak_m_list, acc_y_list, acc_m_list = [], [], [], [], [], [], [], []

    for x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc in loader:
        x_np, y_mm, my_np = x[:, -1, :, :].numpy(), torch.expm1(y_log).clamp_min(0.0).numpy(), my.numpy()
        flash_np, mflash_np, peak_np, mpeak_np, acc_np, macc_np = flash.numpy(), mflash.numpy(), peak.numpy(), mpeak.numpy(), acc.numpy(), macc.numpy()
        B, H, N = y_mm.shape

        for b in range(B):
            for h in range(H):
                mask_h = my_np[b, h, :] > 0.5
                if not mask_h.any(): continue
                X_bh, y_bh = x_np[b], y_mm[b, h, :]
                for j in range(N):
                    if mask_h[j]:
                        X_reg_list.append(X_bh[j]); y_reg_list.append(y_bh[j]); station_reg_list.append(j); lead_reg_list.append(h); mask_reg_list.append(1.0)
            for j in range(N):
                if mflash_np[b, j] > 0.5:
                    X_clf_list.append(x_np[b, j]); station_clf_list.append(j)
                    flash_y_list.append(flash_np[b, j]); peak_y_list.append(peak_np[b, j]); acc_y_list.append(acc_np[b, j])
                    flash_m_list.append(mflash_np[b, j]); peak_m_list.append(mpeak_np[b, j]); acc_m_list.append(macc_np[b, j])

    return {
        "X_reg": np.array(X_reg_list, dtype=np.float32), "y_reg": np.array(y_reg_list, dtype=np.float32), "station_reg": np.array(station_reg_list, dtype=np.int64),
        "X_clf": np.array(X_clf_list, dtype=np.float32), "station_clf": np.array(station_clf_list, dtype=np.int64),
        "flash_y": np.array(flash_y_list, dtype=np.float32), "flash_m": np.array(flash_m_list, dtype=np.float32),
        "peak_y": np.array(peak_y_list, dtype=np.float32), "peak_m": np.array(peak_m_list, dtype=np.float32),
        "acc_y": np.array(acc_y_list, dtype=np.float32), "acc_m": np.array(acc_m_list, dtype=np.float32),
    }

class StationWiseCatBoostBaseline(nn.Module):
    def __init__(self, cat_regressors, cat_flash, cat_peak, cat_acc, num_stations, H_out, quantiles):
        super().__init__()
        self.cat_regressors, self.cat_flash, self.cat_peak, self.cat_acc = cat_regressors, cat_flash, cat_peak, cat_acc
        self.N, self.H_out, self.quantiles = num_stations, H_out, list(quantiles)
        self.K = len(self.quantiles)

    def forward(self, x, regime_id):
        B, T, N, F = x.shape
        x_last = x[:, -1, :, :].detach().cpu().numpy()
        q_mm, flash_prob, peak_prob, acc_prob = np.zeros((B, self.H_out, N, self.K), dtype=np.float32), np.zeros((B, N), dtype=np.float32), np.zeros((B, N), dtype=np.float32), np.zeros((B, N), dtype=np.float32)

        for j in range(N):
            X_j = x_last[:, j, :]
            for k, q in enumerate(self.quantiles):
                model = self.cat_regressors.get((j, q), None)
                q_mm[:, :, j, k] = np.zeros(B, dtype=np.float32)[:, None] if model is None else model.predict(X_j)[:, None]

            m_flash = self.cat_flash.get(j, None)
            flash_prob[:, j] = 0.0 if m_flash is None else m_flash.predict_proba(X_j)[:, 1]

            m_peak = self.cat_peak.get(j, None)
            peak_prob[:, j] = 0.0 if m_peak is None else m_peak.predict_proba(X_j)[:, 1]

            m_acc = self.cat_acc.get(j, None)
            acc_prob[:, j] = 0.0 if m_acc is None else m_acc.predict_proba(X_j)[:, 1]

        q_log = torch.log1p(torch.from_numpy(q_mm).to(device)).clamp_min(0.0)
        def prob_to_logits(p):
            p = torch.clamp(p, 1e-6, 1.0 - 1e-6)
            return torch.log(p / (1.0 - p))
        return q_log, prob_to_logits(torch.from_numpy(flash_prob).to(device)), prob_to_logits(torch.from_numpy(peak_prob).to(device)), prob_to_logits(torch.from_numpy(acc_prob).to(device))

@torch.no_grad()
def evaluate(model, loader, quantiles, thresholds=(0.1, 0.2, 0.3, 0.5)):
    model.eval()
    total_crps_log, total_crps_mm, total_brier_sder, total_brier_peak, total_brier_acc, nb = 0.0, 0.0, 0.0, 0.0, 0.0, 0
    k50 = int(np.argmin(np.abs(np.array(list(quantiles), dtype=np.float32) - 0.5)))
    flash_p_list, flash_y_list, flash_m_list, peak_p_list, peak_y_list, peak_m_list, acc_p_list, acc_y_list, acc_m_list = [], [], [], [], [], [], [], [], []

    for x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc in loader:
        x, reg, y_log, my = x.to(device), reg.to(device), y_log.to(device), my.to(device)
        flash, mflash, peak, mpeak, acc, macc = flash.to(device), mflash.to(device), peak.to(device), mpeak.to(device), acc.to(device), macc.to(device)
        q_hat, flash_logits, peak_logits, acc_logits = model(x, reg)

        total_crps_log += crps_from_quantiles(q_hat, y_log, quantiles, mask=my).item()
        q_mm, y_mm = torch.expm1(q_hat).clamp_min(0.0), torch.expm1(y_log).clamp_min(0.0)
        total_crps_mm += crps_from_quantiles(q_mm, y_mm, quantiles, mask=my).item()

        p_flash, p_peak, p_acc = torch.sigmoid(flash_logits), torch.sigmoid(peak_logits), torch.sigmoid(acc_logits)
        total_brier_sder += brier_score(p_flash, flash, mask=mflash).item()
        total_brier_peak += brier_score(p_peak, peak, mask=mpeak).item()
        total_brier_acc += brier_score(p_acc, acc, mask=macc).item()

        flash_p_list.append(p_flash.detach().cpu().reshape(-1)); flash_y_list.append(flash.detach().cpu().reshape(-1)); flash_m_list.append(mflash.detach().cpu().reshape(-1))
        peak_p_list.append(p_peak.detach().cpu().reshape(-1)); peak_y_list.append(peak.detach().cpu().reshape(-1)); peak_m_list.append(mpeak.detach().cpu().reshape(-1))
        acc_p_list.append(p_acc.detach().cpu().reshape(-1)); acc_y_list.append(acc.detach().cpu().reshape(-1)); acc_m_list.append(macc.detach().cpu().reshape(-1))
        nb += 1

    flash_p, flash_y, flash_m = torch.cat(flash_p_list).numpy(), torch.cat(flash_y_list).numpy(), torch.cat(flash_m_list).numpy()
    peak_p, peak_y, peak_m = torch.cat(peak_p_list).numpy(), torch.cat(peak_y_list).numpy(), torch.cat(peak_m_list).numpy()
    acc_p, acc_y, acc_m = torch.cat(acc_p_list).numpy(), torch.cat(acc_y_list).numpy(), torch.cat(acc_m_list).numpy()

    fm = event_metrics_binary(flash_p, flash_y, flash_m, thresholds=thresholds)
    pm = event_metrics_binary(peak_p, peak_y, peak_m, thresholds=thresholds)
    am = event_metrics_binary(acc_p, acc_y, acc_m, thresholds=thresholds)

    return {
        "CRPS_log": total_crps_log / max(nb, 1),
        "CRPS_mm": total_crps_mm / max(nb, 1),
        "Brier_SDER": total_brier_sder / max(nb, 1),
        "Brier_peak": total_brier_peak / max(nb, 1),
        "Brier_acc": total_brier_acc / max(nb, 1),
        "SDER_metrics": fm,
        "Peak24_metrics": pm,
        "Acc24_metrics": am,
    }

# ============================================================
# 4) CatBoost Training Engines
# ============================================================
def train_station_regressors(X_reg, y_reg, station_reg, params_base, quantiles, num_stations):
    cat_regressors = {}
    for j in range(num_stations):
        mask_j = (station_reg == j)
        if not np.any(mask_j): continue
        X_j, y_j = X_reg[mask_j], y_reg[mask_j]
        if X_j.shape[0] < 50: continue

        for q in quantiles:
            model = CatBoostRegressor(loss_function=f'Quantile:alpha={q}', bootstrap_type='Bernoulli', **params_base)
            model.fit(X_j, y_j, verbose=False)
            cat_regressors[(j, q)] = model
    return cat_regressors

def train_station_classifiers(tab, num_stations, params_base):
    X_clf, station_clf = tab["X_clf"], tab["station_clf"]

    def train_one_target(y, m):
        models = {}
        for j in range(num_stations):
            mask_station = (station_clf == j)
            if not np.any(mask_station): continue
            X_j, y_j, m_j = X_clf[mask_station], y[mask_station], m[mask_station]
            valid = (m_j > 0.5)
            X_j, y_j = X_j[valid], y_j[valid]

            if X_j.shape[0] < 20:
                X_dummy, y_dummy = np.zeros((50, X_clf.shape[1]), dtype=np.float32), np.zeros(50, dtype=np.float32)
                model = CatBoostClassifier(loss_function='Logloss', bootstrap_type='Bernoulli', iterations=10, learning_rate=0.1, depth=4, verbose=False)
                model.fit(X_dummy, y_dummy)
                models[j] = model
                continue

            model = CatBoostClassifier(loss_function='Logloss', bootstrap_type='Bernoulli', **params_base)
            model.fit(X_j, y_j, verbose=False)
            models[j] = model
        return models

    return train_one_target(tab["flash_y"], tab["flash_m"]), train_one_target(tab["peak_y"], tab["peak_m"]), train_one_target(tab["acc_y"], tab["acc_m"])

# ============================================================
# 5) Main Final Training Pipeline
# ============================================================
def load_best_hparams(best_trial_path):
    with open(best_trial_path, "r") as f:
        best = json.load(f)

    raw_q = best.get("quantiles", [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95])
    if isinstance(raw_q, str):
        clean_q = [float(x) for x in raw_q.replace('(', '').replace(')', '').replace('[', '').replace(']', '').split(",")]
    else:
        clean_q = raw_q

    hp = {
        "learning_rate": float(best["learning_rate"]),
        "iterations": int(best["iterations"]),
        "depth": int(best["depth"]),
        "l2_leaf_reg": float(best["l2_leaf_reg"]),
        "subsample": float(best["subsample"]),
        "rsm": float(best["rsm"]),
        "min_data_in_leaf": int(best["min_data_in_leaf"]),
        "quantiles": tuple(clean_q)
    }
    return hp

def final_train_catboost(df_rain, T_in=16, H_out=8, train_frac=0.7, val_frac=0.15):
    best_json_path = os.path.join(TUNING_SAVE_DIR, "best_trial.json")
    print(f"Loading optimal hparams from: {best_json_path}")
    hp = load_best_hparams(best_json_path)
    print("Optimal Hparams:", hp)

    print("\nPre-computing global grid structure...")
    prep, scaler, (thr3h, thrAcc24), (ds_train, ds_val) = build_data_objects(
        df_rain, T_in=T_in, H_out=H_out, train_frac=train_frac, val_frac=val_frac
    )
    N_stations = len(prep["stations"])

    # METHOdOLOGICAL CONFORMITY: Combine Train + Val for final execution
    print(f"Merging Train ({len(ds_train)} rows) and Val ({len(ds_val)} rows) spaces for final execution...")
    ds_final_train = ConcatDataset([ds_train, ds_val])

    # We use a massive batch size for Tabular extraction since CatBoost processes memory natively
    train_loader = DataLoader(ds_final_train, batch_size=2048, shuffle=False)

    print("Flattening Master Dataset for CatBoost Trees...")
    tab_train = build_tabular_from_loader(train_loader)

    params_base = {
        "iterations": hp["iterations"],
        "learning_rate": hp["learning_rate"],
        "depth": hp["depth"],
        "l2_leaf_reg": hp["l2_leaf_reg"],
        "subsample": hp["subsample"],
        "rsm": hp["rsm"],
        "min_data_in_leaf": hp["min_data_in_leaf"],
        "random_seed": 42,
        "verbose": False,
        "thread_count": -1
    }

    print("\n[FINAL CatBoost] Commencing final training of 374 independent Tree Models...")
    print("Training 272 Quantile Regressors...")
    cat_reg = train_station_regressors(tab_train["X_reg"], tab_train["y_reg"], tab_train["station_reg"], params_base, hp["quantiles"], N_stations)

    print("Training 102 Risk Classifiers...")
    cat_flash, cat_peak, cat_acc = train_station_classifiers(tab_train, N_stations, params_base)

    print("\n[FINAL CatBoost] Training completed. Wrapping model and executing final validation score pass...")
    model = StationWiseCatBoostBaseline(cat_reg, cat_flash, cat_peak, cat_acc, N_stations, H_out, hp["quantiles"]).to(device)

    # Evaluate the finalized model on the unified training space to generate the final CSV log
    final_scores = evaluate(model, train_loader, quantiles=hp["quantiles"])

    val_key = float(final_scores["CRPS_log"])
    fm = final_scores["SDER_metrics"]
    pm = final_scores["Peak24_metrics"]
    am = final_scores["Acc24_metrics"]

    # Since CatBoost is not iterative by epoch, we create a single row representing the Final Model State
    row = {
        "final_run": "Complete",
        "train_CRPS_log": val_key,
        "train_CRPS_mm": float(final_scores["CRPS_mm"]),
        "train_Brier_SDER": float(final_scores["Brier_SDER"]),
        "train_Brier_peak": float(final_scores["Brier_peak"]),
        "train_Brier_acc": float(final_scores["Brier_acc"]),
        "SDER_pr_auc": float(fm.get("pr_auc", np.nan)),
        "SDER_roc_auc": float(fm.get("roc_auc", np.nan)),
        "peak_pr_auc": float(pm.get("pr_auc", np.nan)),
        "peak_roc_auc": float(pm.get("roc_auc", np.nan)),
        "acc_pr_auc": float(am.get("pr_auc", np.nan)),
        "acc_roc_auc": float(am.get("roc_auc", np.nan)),
    }

    log_csv = os.path.join(FINAL_SAVE_DIR, "final_training_summary.csv")
    pd.DataFrame([row]).to_csv(log_csv, index=False)

    ckpt_path = os.path.join(FINAL_SAVE_DIR, "final_best_model.pkl")
    joblib.dump({
        "cat_reg": cat_reg,
        "cat_flash": cat_flash,
        "cat_peak": cat_peak,
        "cat_acc": cat_acc,
        "hparams": hp,
        "scaler_mean": getattr(scaler, "mean_", None),
        "scaler_std": getattr(scaler, "std_", None),
        "thr3h": thr3h,
        "thrAcc24": thrAcc24,
        "stations": prep["stations"],
        "times": prep["times"],
        "train_frac": float(train_frac),
        "val_frac": float(val_frac)
    }, ckpt_path)

    print(f"\n✅ [FINAL CatBoost] Complete. Final model architecture generated and evaluated.")
    print("Checkpoint Saved Package:", ckpt_path)
    print("Final State Summary Saved :", log_csv)
    return ckpt_path

# >>> TO EXECUTE RUN THIS COMMAND <<<
if __name__ == "__main__":
    final_train_catboost(df_rain=df_rain, T_in=16, H_out=8, train_frac=0.7, val_frac=0.15)

Using device: cpu
Tuning directory : C:\Users\User\Documents\CatBoost\Tuning_v1
Final save dir   : C:\Users\User\Documents\CatBoost\Final_Run\v2
Loading optimal hparams from: C:\Users\User\Documents\CatBoost\Tuning_v1\best_trial.json
Optimal Hparams: {'learning_rate': 0.01994942877962023, 'iterations': 219, 'depth': 7, 'l2_leaf_reg': 69.58780103230353, 'subsample': 0.6163856702151521, 'rsm': 0.5453032172664104, 'min_data_in_leaf': 81, 'quantiles': (0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95)}

Pre-computing global grid structure...
Merging Train (42929 rows) and Val (9181 rows) spaces for final execution...
Flattening Master Dataset for CatBoost Trees...

[FINAL CatBoost] Commencing final training of 374 independent Tree Models...
Training 272 Quantile Regressors...
Training 102 Risk Classifiers...

[FINAL CatBoost] Training completed. Wrapping model and executing final validation score pass...

✅ [FINAL CatBoost] Complete. Final model architecture generated and evaluated.
Checkpoint Saved 

# **Testing**

In [9]:
# ============================================================
# STANDALONE EVALUATION SCRIPT: STATION-WISE CATBOOST BASELINE
# ============================================================
import os
import random
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score, roc_auc_score

# --- 0) Setup & Paths ---
# Ensure reproducibility
def set_seed(seed=42, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Corrected path to the final model generated by the training script
FINAL_MODELS_PATH = "/content/drive/MyDrive/Rainfall/Updated Research Data/Baselines/CatBoost/Final_Run/v2/final_best_model.pkl"

# --- 1) Utilities ---
class NaNIgnoringStandardScaler:
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def transform(self, X):
        std_safe = np.where((self.std_ is not None) & (self.std_ < self.eps), 1.0, self.std_)
        return (X - self.mean_) / std_safe

def pinball_loss(y_hat_q, y_true, q, mask=None):
    e = y_true - y_hat_q
    loss = torch.maximum((q - 1.0) * e, q * e)
    if mask is not None:
        return (loss * mask).sum() / mask.sum().clamp_min(1.0)
    return loss.mean()

def crps_from_quantiles(y_hat, y_true, quantiles, mask=None):
    return 2.0 * torch.stack([pinball_loss(y_hat[..., k], y_true, q, mask=mask) for k, q in enumerate(quantiles)]).mean()

def brier_score(probs, targets, mask=None):
    loss = (probs - targets) ** 2
    if mask is not None:
        return (loss * mask).sum() / mask.sum().clamp_min(1.0)
    return loss.mean()

def safe_div(a, b, eps=1e-12):
    return a / (b + eps)

def event_metrics_binary(probs, y_true, mask, thresholds=(0.1, 0.2, 0.3, 0.4, 0.5)):
    valid = mask > 0.5
    if valid.sum() < 50:
        return {"n_valid": int(valid.sum()), "pr_auc": np.nan, "roc_auc": np.nan, "by_thr": {thr: {"POD": np.nan, "FAR": np.nan, "CSI": np.nan} for thr in thresholds}}

    p, y = probs[valid], y_true[valid]
    pr_auc = average_precision_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    try:
        roc_auc = roc_auc_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    except:
        roc_auc = np.nan

    by_thr = {}
    for thr in thresholds:
        yhat = (p >= thr).astype(np.float32)
        TP, FP, FN = ((yhat == 1) & (y == 1)).sum(), ((yhat == 1) & (y == 0)).sum(), ((yhat == 0) & (y == 1)).sum()
        by_thr[thr] = {"POD": float(safe_div(TP, TP + FN)), "FAR": float(safe_div(FP, TP + FP)), "CSI": float(safe_div(TP, TP + FP + FN))}

    return {"n_valid": int(valid.sum()), "pr_auc": float(pr_auc) if pr_auc == pr_auc else np.nan, "roc_auc": float(roc_auc) if roc_auc == roc_auc else np.nan, "by_thr": by_thr}

def compute_acc24(Y_rain, M_y, H_out=8):
    T, N = Y_rain.shape
    Acc, Mask = np.full((T, N), np.nan, dtype=np.float32), np.zeros((T, N), dtype=np.float32)
    for t in range(T - H_out):
        window, wmask = Y_rain[t+1:t+1+H_out], M_y[t+1:t+1+H_out]
        ok = (wmask.sum(axis=0) == H_out)
        Mask[t, ok], Acc[t, ok] = 1.0, window[:, ok].sum(axis=0)
    return Acc, Mask

def prepare_full_grid(df_rain):
    df = df_rain.copy()
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.sort_values(["Datetime", "StationID"]).reset_index(drop=True)
    dr_rad = np.deg2rad(df["DR"].astype(np.float32).to_numpy())
    df["DR_sin"], df["DR_cos"] = np.sin(dr_rad), np.cos(dr_rad)

    stations = df[["StationID", "StationName", "Latitude", "Longitude"]].drop_duplicates().sort_values("StationID").reset_index(drop=True)
    stations["node_index"] = np.arange(len(stations))
    N, times = len(stations), np.sort(df["Datetime"].unique())
    T = len(times)

    full_index = pd.MultiIndex.from_product([times, stations["StationID"].values], names=["Datetime", "StationID"])
    meteo_cols = ["DewPointTemperature", "StationLevelPressure", "SP", "Humidity", "Rainfall", "DR_sin", "DR_cos"]
    X_raw = df.set_index(["Datetime", "StationID"])[meteo_cols].sort_index().reindex(full_index).values.reshape(T, N, len(meteo_cols)).astype(np.float32)
    M_in = (~np.isnan(X_raw)).astype(np.float32)
    Y_rain, M_y = X_raw[..., meteo_cols.index("Rainfall")], M_in[..., meteo_cols.index("Rainfall")]

    dt_index = pd.DatetimeIndex(times)
    hour, month = dt_index.hour.astype(np.float32), dt_index.month.astype(np.float32)
    time_feats = np.repeat(np.stack([np.sin(2*np.pi*(hour/24.0)), np.cos(2*np.pi*(hour/24.0)), np.sin(2*np.pi*((month-1)/12.0)), np.cos(2*np.pi*((month-1)/12.0))], axis=-1).astype(np.float32)[:, None, :], N, axis=1)
    season_by_time = df.groupby("Datetime")["Season"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0]).reindex(times).astype(str).values
    unique_seasons = sorted(pd.unique(season_by_time))
    season_to_id = {s: i for i, s in enumerate(unique_seasons)}

    return {"stations": stations, "times": times, "X_raw": X_raw, "M_in": M_in, "Y_rain": Y_rain, "M_y": M_y, "meteo_cols": meteo_cols, "time_feats": time_feats, "season_ids": np.array([season_to_id[s] for s in season_by_time], dtype=np.int64), "unique_seasons": unique_seasons}

class BanglaRainDataset(Dataset):
    def __init__(self, X_scaled, M_in, time_feats, Y_rain, M_y, Acc24, MaskAcc24, season_ids, thr3h, thrAcc24, t_start, t_end, T_in=16, H_out=8, peak_min_obs=None):
        self.X_scaled, self.M_in, self.time_feats = X_scaled, M_in, time_feats
        self.Y_rain, self.M_y, self.Acc24, self.MaskAcc24 = Y_rain, M_y, Acc24, MaskAcc24
        self.season_ids, self.thr3h, self.thrAcc24, self.T_in, self.H_out = season_ids, thr3h, thrAcc24, T_in, H_out
        self.peak_min_obs = peak_min_obs if peak_min_obs is not None else (H_out // 2)
        self.indices = list(range(t_start + (T_in - 1), t_end - H_out))

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]
        x_all = np.concatenate([np.nan_to_num(self.X_scaled[t-self.T_in+1:t+1], nan=0.0).astype(np.float32), self.M_in[t-self.T_in+1:t+1].astype(np.float32), self.time_feats[t-self.T_in+1:t+1]], axis=-1)
        y_win, m_win = self.Y_rain[t+1:t+1+self.H_out], self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        y_log = np.log1p(np.nan_to_num(y_win, nan=0.0)).astype(np.float32)
        m_next = self.M_y[t+1].astype(np.float32)

        flash = ((self.Y_rain[t+1] >= self.thr3h) & (m_next > 0.5)).astype(np.float32)
        mpeak = (m_win.sum(axis=0) >= self.peak_min_obs).astype(np.float32)

        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)

        peak24 = ((y_peak >= self.thr3h) & (mpeak > 0.5)).astype(np.float32)
        macc = self.MaskAcc24[t].astype(np.float32)
        acc24 = ((self.Acc24[t] >= self.thrAcc24) & (macc > 0.5)).astype(np.float32)

        return torch.from_numpy(x_all), torch.tensor(int(self.season_ids[t]), dtype=torch.long), torch.from_numpy(y_log), torch.from_numpy(m_win), torch.from_numpy(flash), torch.from_numpy(m_next), torch.from_numpy(peak24), torch.from_numpy(mpeak), torch.from_numpy(acc24), torch.from_numpy(macc)

# --- 2) Model Wrapper & Eval Function ---
class StationWiseCatBoostBaseline(nn.Module):
    def __init__(self, cat_regressors, cat_flash, cat_peak, cat_acc, num_stations, H_out, quantiles):
        super().__init__()
        self.cat_regressors, self.cat_flash, self.cat_peak, self.cat_acc = cat_regressors, cat_flash, cat_peak, cat_acc
        self.N, self.H_out, self.quantiles = num_stations, H_out, list(quantiles)
        self.K = len(self.quantiles)

    def forward(self, x, regime_id):
        B, T, N, F = x.shape
        x_last = x[:, -1, :, :].detach().cpu().numpy()
        q_mm = np.zeros((B, self.H_out, N, self.K), dtype=np.float32)
        flash_prob = np.zeros((B, N), dtype=np.float32)
        peak_prob = np.zeros((B, N), dtype=np.float32)
        acc_prob = np.zeros((B, N), dtype=np.float32)

        for j in range(N):
            X_j = x_last[:, j, :]
            for k, q in enumerate(self.quantiles):
                model = self.cat_regressors.get((j, q), None)
                q_mm[:, :, j, k] = np.zeros(B, dtype=np.float32)[:, None] if model is None else model.predict(X_j)[:, None]

            m_flash = self.cat_flash.get(j, None)
            flash_prob[:, j] = 0.0 if m_flash is None else m_flash.predict_proba(X_j)[:, 1]

            m_peak = self.cat_peak.get(j, None)
            peak_prob[:, j] = 0.0 if m_peak is None else m_peak.predict_proba(X_j)[:, 1]

            m_acc = self.cat_acc.get(j, None)
            acc_prob[:, j] = 0.0 if m_acc is None else m_acc.predict_proba(X_j)[:, 1]

        q_log = torch.log1p(torch.from_numpy(q_mm).to(device)).clamp_min(0.0)

        def prob_to_logits(p):
            return torch.log(torch.clamp(p, 1e-6, 1.0 - 1e-6) / (1.0 - torch.clamp(p, 1e-6, 1.0 - 1e-6)))

        return q_log, prob_to_logits(torch.from_numpy(flash_prob).to(device)), prob_to_logits(torch.from_numpy(peak_prob).to(device)), prob_to_logits(torch.from_numpy(acc_prob).to(device))

@torch.no_grad()
def evaluate(model, loader, quantiles, thresholds=(0.1, 0.2, 0.3, 0.4, 0.5)):
    model.eval()
    total_crps_log, total_crps_mm, total_brier_sder, total_brier_peak, total_brier_acc, nb = 0.0, 0.0, 0.0, 0.0, 0.0, 0
    k50 = int(np.argmin(np.abs(np.array(list(quantiles), dtype=np.float32) - 0.5)))
    flash_p_list, flash_y_list, flash_m_list = [], [], []
    peak_p_list, peak_y_list, peak_m_list = [], [], []
    acc_p_list, acc_y_list, acc_m_list = [], [], []
    sum_abs, sum_sq, count = None, None, None

    for x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc in loader:
        x, reg, y_log, my = x.to(device), reg.to(device), y_log.to(device), my.to(device)
        flash, mflash, peak, mpeak, acc, macc = flash.to(device), mflash.to(device), peak.to(device), mpeak.to(device), acc.to(device), macc.to(device)

        q_hat, flash_logits, peak_logits, acc_logits = model(x, reg)

        total_crps_log += crps_from_quantiles(q_hat, y_log, quantiles, mask=my).item()
        q_mm, y_mm = torch.expm1(q_hat).clamp_min(0.0), torch.expm1(y_log).clamp_min(0.0)
        total_crps_mm += crps_from_quantiles(q_mm, y_mm, quantiles, mask=my).item()

        p_flash, p_peak, p_acc = torch.sigmoid(flash_logits), torch.sigmoid(peak_logits), torch.sigmoid(acc_logits)

        # Brier flash now maps to SDER metric definition
        total_brier_sder += brier_score(p_flash, flash, mask=mflash).item()
        total_brier_peak += brier_score(p_peak, peak, mask=mpeak).item()
        total_brier_acc += brier_score(p_acc, acc, mask=macc).item()

        q50_mm = torch.expm1(q_hat[..., k50]).clamp_min(0.0)
        B, H, N = y_mm.shape
        if sum_abs is None:
            sum_abs, sum_sq, count = torch.zeros(H, device="cpu"), torch.zeros(H, device="cpu"), torch.zeros(H, device="cpu")

        for h in range(H):
            m = (my[:, h, :] > 0.5)
            if m.any():
                err = (q50_mm[:, h, :][m] - y_mm[:, h, :][m]).detach().cpu()
                sum_abs[h] += err.abs().sum()
                sum_sq[h] += (err**2).sum()
                count[h] += m.sum().detach().cpu()

        flash_p_list.append(p_flash.detach().cpu().reshape(-1)); flash_y_list.append(flash.detach().cpu().reshape(-1)); flash_m_list.append(mflash.detach().cpu().reshape(-1))
        peak_p_list.append(p_peak.detach().cpu().reshape(-1)); peak_y_list.append(peak.detach().cpu().reshape(-1)); peak_m_list.append(mpeak.detach().cpu().reshape(-1))
        acc_p_list.append(p_acc.detach().cpu().reshape(-1)); acc_y_list.append(acc.detach().cpu().reshape(-1)); acc_m_list.append(macc.detach().cpu().reshape(-1))
        nb += 1

    maes, rmses, counts = [], [], []
    for h in range(y_mm.shape[1]):
        if count[h].item() < 1:
            maes.append(np.nan); rmses.append(np.nan); counts.append(0)
        else:
            maes.append(float((sum_abs[h] / count[h]).item()))
            rmses.append(float(torch.sqrt(sum_sq[h] / count[h]).item()))
            counts.append(int(count[h].item()))

    flash_p, flash_y, flash_m = torch.cat(flash_p_list).numpy(), torch.cat(flash_y_list).numpy(), torch.cat(flash_m_list).numpy()
    peak_p, peak_y, peak_m = torch.cat(peak_p_list).numpy(), torch.cat(peak_y_list).numpy(), torch.cat(peak_m_list).numpy()
    acc_p, acc_y, acc_m = torch.cat(acc_p_list).numpy(), torch.cat(acc_y_list).numpy(), torch.cat(acc_m_list).numpy()

    return {
        "CRPS_log": total_crps_log / max(nb, 1),
        "CRPS_mm": total_crps_mm / max(nb, 1),
        "Brier_SDER": total_brier_sder / max(nb, 1),
        "Brier_peak": total_brier_peak / max(nb, 1),
        "Brier_acc": total_brier_acc / max(nb, 1),
        "MAE_mm_by_lead": maes,
        "RMSE_mm_by_lead": rmses,
        "n_valid_by_lead": counts,
        "SDER_metrics": event_metrics_binary(flash_p, flash_y, flash_m, thresholds=thresholds),
        "Peak24_metrics": event_metrics_binary(peak_p, peak_y, peak_m, thresholds=thresholds),
        "Acc24_metrics": event_metrics_binary(acc_p, acc_y, acc_m, thresholds=thresholds),
    }

# --- 3) Execute Test ---
if __name__ == "__main__":
    # Ensure df_rain is loaded here. E.g.:
    # df_rain = pd.read_csv("path_to_your_data.csv")

    print("\nLoading final CatBoost artifacts from:", FINAL_MODELS_PATH)
    art = joblib.load(FINAL_MODELS_PATH)

    # Correctly extracting the nested hyperparameters and models
    hp = art["hparams"]
    quantiles = tuple(hp["quantiles"])
    cat_reg, cat_flash, cat_peak, cat_acc = art["cat_reg"], art["cat_flash"], art["cat_peak"], art["cat_acc"]

    # Extract the exact, frozen scaling and threshold statistics
    scaler_mean = np.array(art["scaler_mean"])
    scaler_std = np.array(art["scaler_std"])
    thr3h = np.array(art["thr3h"])
    thrAcc24 = np.array(art["thrAcc24"])

    # Define structural boundaries (Constant for this model setup)
    T_in = 16
    H_out = 8
    train_frac = float(art["train_frac"])
    val_frac = float(art["val_frac"])

    print("Building full grid from raw data...")
    prep = prepare_full_grid(df_rain)  # Relies on df_rain being globally available
    N_stations, T_total = len(prep["stations"]), len(prep["times"])
    val_end = int(T_total * (train_frac + val_frac))

    # Apply Frozen Scaling perfectly neutralizing data leakage
    print("Applying frozen scaler limits...")
    scaler = NaNIgnoringStandardScaler()
    scaler.mean_, scaler.std_ = scaler_mean, scaler_std

    X_raw = prep["X_raw"]
    X_scaled = scaler.transform(X_raw.reshape(T_total * N_stations, -1)).reshape(X_raw.shape).astype(np.float32)

    Acc24, MaskAcc24 = compute_acc24(prep["Y_rain"], prep["M_y"], H_out)

    # Build testing block
    ds_test = BanglaRainDataset(
        X_scaled, prep["M_in"], prep["time_feats"],
        prep["Y_rain"], prep["M_y"],
        Acc24, MaskAcc24,
        prep["season_ids"],
        thr3h, thrAcc24,
        val_end, T_total, T_in, H_out
    )
    test_loader = DataLoader(ds_test, batch_size=256, shuffle=False)

    model = StationWiseCatBoostBaseline(cat_reg, cat_flash, cat_peak, cat_acc, N_stations, H_out, quantiles).to(device)

    print(f"\nEvaluating CatBoost on TEST split ({len(ds_test)} samples)...")
    with torch.no_grad():
        test_scores = evaluate(model, test_loader, quantiles=quantiles)

    # --- 4) Print Final Metrics ---
    print("\n========== CATBOOST STATION-WISE TEST METRICS ==========")
    print(f"CRPS_log   : {test_scores['CRPS_log']:.4f}")
    print(f"CRPS_mm    : {test_scores['CRPS_mm']:.4f}")
    print(f"Brier_SDER : {test_scores['Brier_SDER']:.4f}")
    print(f"Brier_peak : {test_scores['Brier_peak']:.4f}")
    print(f"Brier_acc  : {test_scores['Brier_acc']:.4f}")

    fm, pm, am = test_scores["SDER_metrics"], test_scores["Peak24_metrics"], test_scores["Acc24_metrics"]

    print("\n[SDER 3h]   PR-AUC={:.4f}, ROC-AUC={:.4f}, n_valid={}".format(fm["pr_auc"], fm["roc_auc"], fm["n_valid"]))
    if 0.5 in fm["by_thr"]:
        m = fm["by_thr"][0.5]
        print(f"            @Thr=0.5: CSI={m['CSI']:.4f}, POD={m['POD']:.4f}, FAR={m['FAR']:.4f}")

    print("[Peak24]    PR-AUC={:.4f}, ROC-AUC={:.4f}, n_valid={}".format(pm["pr_auc"], pm["roc_auc"], pm["n_valid"]))
    if 0.5 in pm["by_thr"]:
        m = pm["by_thr"][0.5]
        print(f"            @Thr=0.5: CSI={m['CSI']:.4f}, POD={m['POD']:.4f}, FAR={m['FAR']:.4f}")

    print("[Acc24]     PR-AUC={:.4f}, ROC-AUC={:.4f}, n_valid={}".format(am["pr_auc"], am["roc_auc"], am["n_valid"]))
    if 0.5 in am["by_thr"]:
        m = am["by_thr"][0.5]
        print(f"            @Thr=0.5: CSI={m['CSI']:.4f}, POD={m['POD']:.4f}, FAR={m['FAR']:.4f}")

    print("\nMAE_mm_by_lead :", [round(val, 4) if not np.isnan(val) else "NaN" for val in test_scores['MAE_mm_by_lead']])
    print("RMSE_mm_by_lead:", [round(val, 4) if not np.isnan(val) else "NaN" for val in test_scores['RMSE_mm_by_lead']])

Using device: cpu

Loading final CatBoost artifacts from: /content/drive/MyDrive/Rainfall/Updated Research Data/Baselines/CatBoost/Final_Run/v2/final_best_model.pkl
Building full grid from raw data...
Applying frozen scaler limits...

Evaluating CatBoost on TEST split (9181 samples)...

========== CATBOOST STATION-WISE TEST METRICS ==========
CRPS_log   : 0.1096
CRPS_mm    : 0.5833
Brier_SDER : 0.0337
Brier_peak : 0.1131
Brier_acc  : 0.0364

[SDER 3h]   PR-AUC=0.3508, ROC-AUC=0.8839, n_valid=308560
            @Thr=0.5: CSI=0.1058, POD=0.1123, FAR=0.3550
[Peak24]    PR-AUC=0.5518, ROC-AUC=0.8537, n_valid=308571
            @Thr=0.5: CSI=0.2402, POD=0.2806, FAR=0.3745
[Acc24]     PR-AUC=0.2084, ROC-AUC=0.8451, n_valid=308427
            @Thr=0.5: CSI=0.0165, POD=0.0166, FAR=0.3793

MAE_mm_by_lead : [0.6607, 0.6654, 0.6674, 0.6685, 0.6693, 0.6696, 0.6701, 0.6702]
RMSE_mm_by_lead: [3.9778, 3.9868, 3.9906, 3.9928, 3.9936, 3.9942, 3.9948, 3.9955]
